# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [2]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [3]:

df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Units: {total_units:,}")

Total Revenue: $8,520.00
Total Units: 783


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [5]:
by_category = (
    df.groupby('category', as_index=False)['revenue']
    .sum()
    .sort_values(by='revenue', ascending=False)
)

by_category['share_pct'] = (by_category['revenue'] / df['revenue'].sum()) * 100

print(by_category)

   category  revenue  share_pct
1      Food   4293.0  50.387324
2     Merch   1771.5  20.792254
0     Drink   1554.0  18.239437
3  RainGear    901.5  10.580986


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [7]:
vendor_stats = df.groupby('vendor_id').agg(
    avg_order_revenue=('revenue', 'mean'),
    order_count=('revenue', 'count')
).sort_values(by='avg_order_revenue', ascending=False)

print(vendor_stats)

           avg_order_revenue  order_count
vendor_id                                
V-01               22.595745           94
V-18               21.750000          108
V-05               20.580645           93
V-10               20.314286          105


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [11]:
merch_rev = df.loc[df['category'] == 'Merch', 'revenue'].sum()
merch_share = (merch_rev / df['revenue'].sum()) * 100

print(f"{merch_share:.1f}% comes from merch")

20.8% comes from merch


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [13]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

assert len(joined) == len(df), "Row count changed!"
assert abs(joined['revenue'].sum() - df['revenue'].sum()) < 1e-6, "Revenue total changed!"

unmatched = joined.loc[joined['vendor_name'].isna(), 'vendor_id'].unique()
print(f"Unmatched vendor ID(s): {list(unmatched)}")

joined['vendor_name'] = joined['vendor_name'].fillna('Unknown')

print(f"Merge successful: {len(joined)} rows, total revenue ${joined['revenue'].sum():,.2f}")

Unmatched vendor ID(s): ['V-18']
Merge successful: 400 rows, total revenue $8,520.00


**The unmatched vendor, and what I did about it:** _..._

V-18 was in the orders data but not in `vendor_names`. Using a left join keeps all 400 of the order rows, and the full revenue total. However, I replaced the missing name of the vendor with "Unknown"

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [14]:
pivot_report = pd.pivot_table(
    joined,
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

print(pivot_report)

category          Drink    Food   Merch  RainGear   Total
vendor_name                                              
Cav Merch North   502.5  1054.5   400.5     175.5  2133.0
Hoos Burgers      171.0  1338.0   373.5     241.5  2124.0
Rotunda Tacos     298.5   882.0   489.0     244.5  1914.0
Unknown           582.0  1018.5   508.5     240.0  2349.0
Total            1554.0  4293.0  1771.5     901.5  8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [15]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.



### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

Vendors should probably focus more on selling food than anything else, as they dominate revenue (50.4% of all revenue). More specifically however, it seems that all vendors sell all products (Hoos Burgers sell rain gear, and Cav Merch North sells food). Specializing in one or the other could increase overall revenue and all for better margins. Burgers also seem to sell better than tacos (~1300 dollars vs ~800 dollars), so maybe the taco stand should look to add more variety.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

For Q5 and Q6, my answers and generated reports depend on V-18, the unkown vendor. This makes recommendations based on these reports untrustworthy, as we don't actaully know what V-18 is. Is it just a vendor whose name wasn't inputted? Is it a placeholder for "all other vendors"? Is it a pop-up shop? This vendor also generated the most revenue however, so its and important irregularity.
